# PACE ocean-colour chlorophyll (NASA Earthdata)

Download a daily PACE OCI Level-3 mapped chlorophyll-a granule from the OB.DAAC and map it on a log scale. This is a gridded **raster** product, fetched through one Earthdata Login.

> Live query — needs the `[earthdata]` extra and EDL credentials. Each fetch and plot step is wrapped in a `try`/`except` so the notebook degrades gracefully (nbval-lax safe) when run offline or without credentials.

## Setup

Import the unified `EarthLens` entry point and prepare a local output directory for the downloaded granule.

In [ ]:
from pathlib import Path

from earthlens.core import EarthLens

OUT_DIR = Path('earthdata_output')
OUT_DIR.mkdir(exist_ok=True)

## 1 · Download the chlorophyll granule

Earthdata is a **raster** backend, so `download()` writes the granule(s) to disk and returns the list of written paths. We build the request first — the PACE OCI Level-3 mapped chlorophyll dataset, the `chlor_a` variable, a single day, and a global-ish bounding box.

In [ ]:
ocean = EarthLens(
    data_source='earthdata',
    dataset='PACE_OCI_L3M_CHL_31',
    variables=['chlor_a'],
    start='2024-06-01',
    end='2024-06-01',
    aoi=[-180.0, -60.0, 180.0, 60.0],
    path=str(OUT_DIR),
)

The download is run in its own step, wrapped in a `try`/`except` so the notebook keeps going (printing a skip message) when EDL credentials or the `[earthdata]` extra are unavailable.

In [ ]:
paths = None
try:
    paths = ocean.download(progress_bar=False)
    print(len(paths), 'granule(s):', [Path(p).name for p in paths])
except Exception as exc:
    print(f'skipped live query: {type(exc).__name__}: {exc}')

## 2 · Map the chlorophyll field

Open the granule with `xarray` and plot `chlor_a` on a logarithmic colour scale — chlorophyll spans several orders of magnitude, so `LogNorm` keeps both clear open-ocean water and productive coastal blooms legible. The plot is guarded by `if paths:` and its own `try`/`except` so it is skipped cleanly when the download did not run.

In [ ]:
if paths:
    try:
        import matplotlib.pyplot as plt
        import xarray as xr
        from matplotlib.colors import LogNorm

        ds = xr.open_dataset(paths[0])
        chl = ds['chlor_a']
        fig, ax = plt.subplots(figsize=(10, 5))
        chl.plot(ax=ax, norm=LogNorm(vmin=0.01, vmax=20), cmap='viridis')
        ax.set_title('PACE OCI chlorophyll-a, 2024-06-01')
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print(f'plot step skipped: {type(exc).__name__}: {exc}')